In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
import requests
import os

# Initialize FastAPI
app = FastAPI()

# Configuration - set your API key here or via environment variable
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "sk-2954077ef6724ad5a51b6dcdb2ad07fe")
DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"  # Example endpoint, adjust if different

# Define request/response models
class Message(BaseModel):
    role: str
    content: str

class InferenceRequest(BaseModel):
    messages: list[Message]  # DeepSeek API typically uses a chat format
    max_tokens: int = 100
    model: str = "deepseek-chat"  # or whatever model name DeepSeek provides

class InferenceResponse(BaseModel):
    generated_text: str

# Inference endpoint
@app.post("/infer", response_model=InferenceResponse)
async def infer(request: InferenceRequest):
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": request.model,
        "messages": [msg.dict() for msg in request.messages],
        "max_tokens": request.max_tokens
    }
    
    try:
        response = requests.post(DEEPSEEK_API_URL, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        
        # Extract the generated text from the response
        # Adjust this based on the actual API response structure
        generated_text = result["choices"][0]["message"]["content"]
        
        return {"generated_text": generated_text}
    except requests.exceptions.RequestException as e:
        return {"generated_text": f"Error calling DeepSeek API: {str(e)}"}

# Run with: uvicorn main:app --reload

In [2]:
!uvicorn main:app --reload


INFO:     Will watch for changes in these directories: ['/workspace/moataz-work/infer_api']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [575] using WatchFiles
Process SpawnProcess-1:
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.10/dist-packages/uvicorn/_subprocess.py", line 80, in subprocess_started
    target(sockets=sockets)
  File "/usr/local/lib/python3.10/dist-packages/uvicorn/server.py", line 65, in run
    return asyncio.run(self.serve(sockets=sockets))
  File "/usr/lib/python3.10/asyncio/runners.py", line 44, in run
    return loop.run_until_complete(main)
  File "uvloop/loop.pyx", line 1517, in uvloop.loop.Loop.run_until_complete
  File "/usr/local/lib/python3.10/dist-packages/uvi


KeyboardInterrupt

